# Balanced XGBoost hyperparameter comparison

All five runs select 25 features from the training set, use the training class ratio for `scale_pos_weight`, and score the validation set at a 0.5 decision threshold. The test set is reserved for final evaluation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn

with np.load("data/processed_datasets.npz") as processed_data:
    X_train = pd.DataFrame(processed_data["X_train"])
    X_val = pd.DataFrame(processed_data["X_val"])
    Y_train = pd.Series(processed_data["Y_train"], name="churn")
    Y_val = pd.Series(processed_data["Y_val"], name="churn")

N_FEATURES = 25
THRESHOLD = 0.5
assert X_train.shape[1] >= N_FEATURES
positive_class_weight = (Y_train == 0).sum() / (Y_train == 1).sum()
print("Training:", X_train.shape, "Validation:", X_val.shape)
print("XGBoost positive class weight:", positive_class_weight)

mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("churn-xgboost-balanced-hyperparameters")

/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training: (659028, 44) Validation: (82379, 44)
XGBoost positive class weight: 9.085670997658509


<Experiment: artifact_location='/home/rami/projects/mlops/mlruns/5', creation_time=1790616968343, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1790676198597, lifecycle_stage='active', name='churn-xgboost-balanced-hyperparameters', tags={}, trace_location=None, workspace='default'>

In [ ]:
param_configs = [
    {
        "learning_rate": 0.03,
        "max_depth": 3,
        "n_estimators": 400,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.05,
        "max_depth": 3,
        "n_estimators": 400,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.05,
        "max_depth": 5,
        "n_estimators": 400,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.1,
        "max_depth": 5,
        "n_estimators": 200,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    {
        "learning_rate": 0.1,
        "max_depth": 7,
        "n_estimators": 200,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
]

In [3]:
def evaluate_probabilities(y, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }


results = []
for config_number, params in enumerate(param_configs, start=1):
    run_name = f"balanced_xgboost_top_{N_FEATURES}_config_{config_number}"
    print("Running:", run_name, params, flush=True)
    pipeline = Pipeline([
        ("feature_selection", SelectKBest(score_func=f_classif, k=N_FEATURES)),
        ("model", XGBClassifier(
            **params,
            scale_pos_weight=positive_class_weight,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1,
        )),
    ])

    with mlflow.start_run(run_name=run_name):
        pipeline.fit(X_train, Y_train)
        probabilities = pipeline.predict_proba(X_val)[:, 1]
        metrics = evaluate_probabilities(Y_val, probabilities, THRESHOLD)
        mlflow.log_params({
            "model_type": "xgboost",
            "balance": "balanced",
            "n_features": N_FEATURES,
            "threshold": THRESHOLD,
            "evaluation_split": "validation",
            "scale_pos_weight": positive_class_weight,
            **params,
        })
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            pipeline, name="model", serialization_format="cloudpickle"
        )

    results.append({
        "config": config_number,
        **params,
        "n_features": N_FEATURES,
        "threshold": THRESHOLD,
        **metrics,
    })
    print("Finished:", run_name, "F1:", round(metrics["f1"], 4), flush=True)

Running: balanced_xgboost_top_25_config_1 {'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8, 'colsample_bytree': 0.8}


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:03:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: balanced_xgboost_top_25_config_1 F1: 0.2541
Running: balanced_xgboost_top_25_config_2 {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8, 'colsample_bytree': 0.8}


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:04:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: balanced_xgboost_top_25_config_2 F1: 0.254
Running: balanced_xgboost_top_25_config_3 {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 400, 'subsample': 0.8, 'colsample_bytree': 0.8}


...

Finished: balanced_xgboost_top_25_config_3 F1: 0.2547
Running: balanced_xgboost_top_25_config_4 {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8, 'colsample_bytree': 0.8}


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:04:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: balanced_xgboost_top_25_config_4 F1: 0.2542
Running: balanced_xgboost_top_25_config_5 {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8, 'colsample_bytree': 0.8}


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 11:04:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: balanced_xgboost_top_25_config_5 F1: 0.2543


In [4]:
results_df = pd.DataFrame(results).sort_values(
    by="f1", ascending=False
).reset_index(drop=True)
results_df.to_csv("data/xgboost_hyperparameter_results.csv", index=False)
results_df

,config,learning_rate,max_depth,n_estimators,subsample,colsample_bytree,n_features,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,3,0.05,5,400,0.8,0.8,25,0.5,0.630452,0.633310,0.159185,0.636876,0.254707,0.682289,0.203788
1,5,0.10,7,200,0.8,0.8,25,0.5,0.648151,0.629024,0.160994,0.605167,0.254328,0.676919,0.196843
2,4,0.10,5,200,0.8,0.8,25,0.5,0.631035,0.632435,0.158959,0.634182,0.254202,0.682100,0.202994
3,1,0.03,3,400,0.8,0.8,25,0.5,0.625803,0.633454,0.158374,0.642997,0.254150,0.684079,0.206715
4,2,0.05,3,400,0.8,0.8,25,0.5,0.626786,0.633019,0.158390,0.640793,0.253998,0.684020,0.206391
